#**Chapter7. 앙상블 학습과 랜덤 포레스트**

일련의 예측기로부터 예측을 수집하면 가장 좋은 모델 하나보다 더 좋은 예측을 얻을 수 있을 것. 일련의 예측기를 앙상블이라고 부르고 이를 앙상블 학습이라고 함.

이 장에서는 배깅, 부스팅, 스태킹 등 가장 인기 있는 앙상블 방법을 설명함.

##**7.1. 투표 기반 분류기**
가장 좋은 분류기를 만드는 매우 간단한 방법은 각 분류기의 예측을 모아서 가장 많이 선택된 클래스를 예측하는 것. => 직접 투표 분류기.

이 다수결 투표 분류기가 앙상블에 포함된 개별 분류기 중 가장 뛰어난 것보다 정확도가 높을 경우가 많음. 큰수의 법칙 때문에.

In [2]:
import warnings
warnings.filterwarnings('ignore')

# import package
import numpy as np
import os

#5장에서 소개한 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
X,y = make_moons(n_samples=100, noise=0.15)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn. linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf = LogisticRegression()
rnd_clf = RandomForestClassifier()
svm_clf = SVC()

voting_clf = VotingClassifier(
    estimators=[('lr',log_clf),('rf',rnd_clf),('svc',svm_clf)],
    voting='hard')
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [4]:
from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.85
RandomForestClassifier 0.95
SVC 1.0
VotingClassifier 1.0


##**7.2. 배깅과 페이스팅**

분류기를 만드는 한가지 방법은 각기 다른 훈련 알고리즘을 사용하는 것이고, 또 다른 방법은 같은 알고리즘을 사용하고 훈련 세트의 세브 셋을 무작위로 구성하여 분류기를 각기 다르게 학습시키는 것.

훈련세트에서 중복을 허용하여 샘플링하는 방식을 배깅, 중복을 허용하지 않고 샘플링하는 방식을 페이스팅이라고 함.

###7.2.1. 사이킷런의 배깅과 페이스팅

사이킷런은 배깅과 페이스팅을 위해 BaggingClassifier을 제공.

In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf = BaggingClassifier(
DecisionTreeClassifier(), n_estimators=500,
max_samples=50, bootstrap=True, n_jobs=-1)
bag_clf.fit(X_train, y_train)
y_pred = bag_clf.predict(X_test)

###7.2.2. oob 평가

배깅을 사용하면 어떤 샘플은 한 예측기를 위해 여러 번 샘플링되고 어떤 것은 전혀 선택되지 않음. BaggingClassifier는 기본값으로 중복을 허용하여 훈련 세트의 크기만큼인 m개 샘플을 선택.

선택되지 않은 훈련 샘플의 나머지 37%를 oob 샘플이라고 부름.

In [7]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    bootstrap=True, n_jobs=-1, oob_score=True)

bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.95

In [8]:
from sklearn.metrics import top_k_accuracy_score
y_pred = bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.95

In [9]:
bag_clf.oob_decision_function_

array([[0.53038674, 0.46961326],
       [0.03954802, 0.96045198],
       [0.94535519, 0.05464481],
       [0.95698925, 0.04301075],
       [0.        , 1.        ],
       [0.24193548, 0.75806452],
       [0.99450549, 0.00549451],
       [0.43558282, 0.56441718],
       [0.        , 1.        ],
       [1.        , 0.        ],
       [0.99456522, 0.00543478],
       [1.        , 0.        ],
       [0.00555556, 0.99444444],
       [0.        , 1.        ],
       [0.18085106, 0.81914894],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.98387097, 0.01612903],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.84946237, 0.15053763],
       [0.73184358, 0.26815642],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.02051282, 0.97948718],
       [1.        , 0.        ],
       [0.00552486, 0.99447514],
       [0.25698324, 0.74301676],
       [1.        , 0.        ],
       [0.

##**7.3. 랜덤 패치와 랜덤 서브스페이스**

BaggingClassifier는 특성샘플링도 지원. 샘플링은 max_festures, bootstrap_features두 매개변수로 조절.

이 기법은 특히 이미지와 같은 매우  고차원의 데이터 셋을 다룰 때 유용. 훈련 특성과 샘플을 모두 샘플링하는 것을 랜덤 패치 방식이라고 함.

##**7.4. 랜덤 포레스트**

랜덤 포레스트는 일반적으로 배깅 방법을 적용한 결정 트리의 앙상블.

In [10]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf = RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(X_train, y_train)

y_pred_rf = rnd_clf.predict(X_test)

In [12]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(max_features="auto",max_leaf_nodes=16),
    n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1)

###7.4.1. 엑스트라 트리

이와 같이 극단적으로 무작위한 트리의 랜덤 포레스트를 익스트림 랜덤 트리 앙상블이라고 부름.

###7.4.2. 특성 중요도

랜덤 포레스트의 또 다른 장점은 특성의 상대적 중요도를 측정하기 쉽다는 것. 사이킷런은 어떤 특성을 사용한 노드가 평균적으로 불순도를 얼마나 감소시키는지 확인하여 특성의 중요도를 측정.

In [14]:
from sklearn.datasets import load_iris
iris = load_iris()
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris["data"],iris["target"])
for name, score in zip(iris["feature_names"], rnd_clf.feature_importances_):
    print(name, score)

sepal length (cm) 0.10752043610445687
sepal width (cm) 0.02630222842452641
petal length (cm) 0.44654914633331605
petal width (cm) 0.4196281891377007


##**7.5. 부스팅**

부스팅은 약한 학습기를 여러 개 연결하여 강한 학습기를 만드는 앙상블 방법을 말함. 부스팅 방법에는 에이다부스트와 그레이디언트 부스팅이 있음.

###7.5.1. 에이다부스트

이전 예측기를 보완하는 새로운 예측기를 만드는 방법은 이전 모델이 과소적합했던 훈련 샘플의 가중치를 더 높이는 것. 새로운 예측기는 학습하기 어려운 샘플에 점점 더 맞춰지게 됨.

모든 예측기가 훈련을 마치면 이 앙상블은 배깅이나 페이스팅과 비슷한 방식으로 예측을 만듦. 하지만 가중치가 적용된 훈련 세트의 전반적인 정확도에 따라 예측기마다 다른 가중치가 적용됨.

In [15]:
from sklearn.ensemble import AdaBoostClassifier
ada_clf = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=1), n_estimators=200,
    algorithm="SAMME.R", learning_rate=0.5)
ada_clf.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                   learning_rate=0.5, n_estimators=200)

###7.5.2. 그레이디언트 부스팅

인기가 높은 또 하나의 부스팅 알고리즘은 그레이디언트 부스팅. 에이다부스트처럼 반복마다 샘플의 가중치를 수정하는 그레이디언트 부스팅은 앙상블에 이전까지의 오차를 보정하도록 예측기를 순차적으로 추가.

In [16]:
from sklearn.tree import DecisionTreeRegressor

tree_reg1 = DecisionTreeRegressor(max_depth=2)
tree_reg1.fit(X,y)

DecisionTreeRegressor(max_depth=2)

In [18]:
y2 = y - tree_reg1.predict(X)
tree_reg2 = DecisionTreeRegressor(max_depth=2)
tree_reg2.fit(X,y2)

DecisionTreeRegressor(max_depth=2)

In [20]:
y3 = y2 - tree_reg2.predict(X)
tree_reg3 = DecisionTreeRegressor(max_depth=2)
tree_reg3.fit(X,y3)

DecisionTreeRegressor(max_depth=2)

In [22]:
#y_pred = sum(tree.predict(X_new) for tree in (tree_reg1, tree_reg2, tree_reg3))

In [23]:
from sklearn.ensemble import GradientBoostingRegressor

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=3, learning_rate=1.0)
gbrt.fit(X,y)

GradientBoostingRegressor(learning_rate=1.0, max_depth=2, n_estimators=3)

learning_rate 매개변수가 각 트리의 기여 정도를 조절하는데, 이를 낮게 설정하면 앙상블을 훈련 세트에 학습시키기 위해 많은 트리가 필요하지만 일반적으로 예측의 성능은 좋아짐. 이는 축소라고 부르는 규제 방법.

In [24]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train,X_val, y_train, y_val = train_test_split(X,y)

gbrt = GradientBoostingRegressor(max_depth=2, n_estimators=120)
gbrt.fit(X_train, y_train)

errors = [mean_squared_error(y_val, y_pred)
        for y_pred in gbrt.staged_predict(X_val)]
bst_n_estimators = np.argmin(errors) + 1

gbrt_best = GradientBoostingRegressor(max_depth=2, n_estimators=bst_n_estimators)
gbrt_best.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=2, n_estimators=120)

In [25]:
gbrt = GradientBoostingRegressor(max_depth=2, warm_start=True)

min_val_error = float("inf")
error_going_up = 0
for n_estimators in range(1,120):
    gbrt.n_estimators = n_estimators
    gbrt.fit(X_train, y_train)
    y_pred = gbrt.predict(X_val)
    val_error = mean_squared_error(y_val, y_pred)
    if val_error < min_val_error:
        min_val_error = val_error
        error_going_up = 0
    else:
        error_going_up += 1
        if error_going_up == 5:
            break

GradientBoostingRegressor 는 각 트리가 훈련할 때 사용할 훈련 샘플의 비율을 지정할 수 있는 subsample 매개변수도 지원. 편향이 높아지는 대신 분산이 낮아지게 됨. 확률적 그레이디언트 부스팅이라고 함.

In [26]:
import xgboost

xgb_reg = xgboost.XGBRegressor()
xgb_reg.fit(X_train, y_train)
y_pred = xgb_reg.predict(X_val)

In [28]:
xgb_reg.fit(X_train, y_train, eval_set = [(X_val,y_val)], early_stopping_rounds=2)
y_pred = xgb_reg.predict(X_val)

TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'

##**7.6. 스태킹**

앙상블에 속한 모든 예측기의 예측을 취합하는 간단함 함수를 사용하는 대신 취합하는 모델을 훈련시킬 수 없을까